In [18]:
import json
import os
import random
import re
from pathlib import Path
import pandas as pd

# ==========================================
# ⚙️ 配置区域 (在这里修改你的路径和参数)
# ==========================================
INPUT_PATH = "/mnt/data/zwl/verl/data/mixed_40.jsonl"       # 源 JSONL 文件路径
OUTPUT_DIR = ""                  # parquet 文件的保存目录
OUTPUT_BASENAME = "mixed_40_grpo"   # 生成文件的基础名称
VAL_RATIO = 0.1                     # 验证集划分比例 (设为 0 则不划分)
SEED = 42                           # 随机种子

# ==========================================
# 🛠️ 核心函数定义
# ==========================================
def load_jsonl(path: str) -> list[dict]:
    dataset = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                dataset.append(json.loads(line))
    return dataset

def extract_last_nonempty_tag(text: str, tag: str) -> str:
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text or "", flags=re.DOTALL | re.IGNORECASE)
    for candidate in reversed(matches):
        candidate = candidate.strip()
        if candidate:
            return candidate
    return ""

def normalize_output_tags(text: str) -> str:
    """
    Keep at most one think block and one final block in a predictable order.
    This is only for metadata/debugging; reward ground_truth still comes from <final>.
    """
    text = text or ""
    think = extract_last_nonempty_tag(text, "think")
    final = extract_last_nonempty_tag(text, "final")

    if final:
        if think:
            return f"<think>\n{think}\n</think>\n<final>\n{final}\n</final>"
        return f"<final>\n{final}\n</final>"

    if think:
        return f"<think>\n{think}\n</think>"

    return text.strip()

def build_prompt(record: dict) -> list[dict]:
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    user_content = instruction + ("\n" + input_text if input_text else "")
    return [{"role": "user", "content": user_content}]

def convert_records(raw_dataset: list[dict], data_source: str) -> list[dict]:
    converted_rows = []
    for idx, item in enumerate(raw_dataset):
        raw_output = str(item.get("output", ""))
        ground_truth = extract_last_nonempty_tag(raw_output, "final")
        
        if not ground_truth:
            raise ValueError(f"Record {idx} has no non-empty <final>...</final> in output.")

        converted_rows.append(
            {
                "uid": idx,
                "data_source": data_source,
                "ability": "reasoning",
                "prompt": build_prompt(item),
                "messages": build_prompt(item),
                "ground_truth": ground_truth,
                "reward_model": {
                    "style": "rule",
                    "ground_truth": ground_truth,
                },
                "extra_info": {
                    "index": idx,
                    "difficulty": item.get("difficulty", ""),
                    "view": item.get("view", ""),
                },
                "raw_output": raw_output,
                "normalized_output": normalize_output_tags(raw_output),
            }
        )
    return converted_rows

def save_parquet(rows: list[dict], path: Path) -> None:
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

# ==========================================
# 🚀 运行逻辑
# ==========================================
def process_data():
    raw_dataset = load_jsonl(INPUT_PATH)
    if not raw_dataset:
        raise ValueError(f"Input dataset is empty: {INPUT_PATH}")

    data_source = Path(INPUT_PATH).stem
    converted_rows = convert_records(raw_dataset, data_source=data_source)

    output_dir = Path(OUTPUT_DIR)
    full_path = output_dir / f"{OUTPUT_BASENAME}.parquet"
    save_parquet(converted_rows, full_path)

    print(f"Loaded {len(raw_dataset)} examples from: {INPUT_PATH}")
    print(f"Saved full parquet to: {full_path}")

    if VAL_RATIO and VAL_RATIO > 0:
        if not 0 < VAL_RATIO < 1:
            raise ValueError("VAL_RATIO must be in (0, 1) when enabled.")
        rng = random.Random(SEED)
        shuffled = converted_rows[:]
        rng.shuffle(shuffled)

        val_size = max(1, int(round(len(shuffled) * VAL_RATIO)))
        val_rows = shuffled[:val_size]
        train_rows = shuffled[val_size:]

        train_path = output_dir / f"{OUTPUT_BASENAME}_train.parquet"
        val_path = output_dir / f"{OUTPUT_BASENAME}_val.parquet"
        save_parquet(train_rows, train_path)
        save_parquet(val_rows, val_path)

        print(f"Saved train parquet ({len(train_rows)} rows) to: {train_path}")
        print(f"Saved val parquet ({len(val_rows)} rows) to: {val_path}")

# 执行数据处理
process_data()

正在读取并清洗数据: /mnt/data/zwl/verl/data/mixed_40.jsonl
正在划分数据集 (验证集比例: 0.1)...


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1868.29ba/s]

✅ 处理完成！
📦 训练集 (36 条) -> /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet
📦 验证集 (4 条) -> /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_val.parquet


In [19]:
import pyarrow.parquet as pq

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet')

# 转换为 pandas DataFrame
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")

行数: 36
列名: ['prompt', 'ground_truth', 'reward_model', 'raw_output', 'difficulty', 'view']


In [20]:
import pyarrow.parquet as pq
import pandas as pd
import json

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet')
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print(f"数据形状: {df.shape}")
print("\n" + "="*80)
print("前 1 条数据（完整内容）:")
print("="*80)

# 设置 pandas 显示选项，显示完整内容
pd.set_option('display.max_colwidth', None)  # 不限制列宽
pd.set_option('display.max_rows', None)       # 不限制行数
pd.set_option('display.width', None)          # 不限制宽度
pd.set_option('display.max_seq_items', None)  # 不限制序列项

# 方法1: 直接显示
print(df.head(1))

print("\n" + "="*80)
print("逐列详细查看:")
print("="*80)

# 方法2: 逐列显示完整内容
for col in df.columns:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    print(f"前1条内容:")
    try:
        # 尝试美化显示
        value = df[col].iloc[0]
        if isinstance(value, (dict, list)):
            print(json.dumps(value, ensure_ascii=False, indent=2))
        else:
            print(value)
    except Exception as e:
        print(f"无法显示: {e}")

print("\n" + "="*80)
print("数据统计信息:")
print("="*80)
print(df.describe())

# 如果有字符串列，显示长度信息
print("\n" + "="*80)
print("字符串列长度信息:")
print("="*80)
for col in df.select_dtypes(include=['object']).columns:
    try:
        print(f"{col}: 最小长度={df[col].str.len().min()}, 最大长度={df[col].str.len().max()}, 平均长度={df[col].str.len().mean():.2f}")
    except:
        pass

行数: 36
列名: ['prompt', 'ground_truth', 'reward_model', 'raw_output', 'difficulty', 'view']
数据形状: (36, 6)

前 1 条数据（完整内容）:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

/tmp/ipykernel_15404/3188049470.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
